In [1]:
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score



print("=" * 70)
print("STEP 0: REBUILD THE SAME DATA PIPELINE AS PART 3 (BASIC MODELING)")
print("=" * 70)

df = pd.read_csv('cleaned_data.csv')

y_clf = (df['HeartDisease'] == 'Yes').astype(int)
X_raw = df.drop(columns=['PatientID', 'Cholesterol', 'HeartDisease', 'DiagnosisYear'])

categorical_cols = ['Sex', 'Smoker', 'BloodType', 'FamilyHistory', 'Diabetes']
X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True)

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print(f"Training rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}, Features: {X.shape[1]}")


print("\n" + "=" * 70)
print("TASK 1: DECISION TREE BASELINE (UNCONSTRAINED)")
print("=" * 70)

dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train_scaled, y_clf_train)

train_acc_baseline = accuracy_score(y_clf_train, dt_baseline.predict(X_train_scaled))
test_acc_baseline = accuracy_score(y_clf_test, dt_baseline.predict(X_test_scaled))

print(f"Training accuracy: {train_acc_baseline:.4f}")
print(f"Test accuracy:     {test_acc_baseline:.4f}")
print(f"Train-test gap:    {train_acc_baseline - test_acc_baseline:.4f}")


print("\n" + "=" * 70)
print("TASK 2: CONTROLLED DECISION TREE")
print("=" * 70)

dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

train_acc_controlled = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
test_acc_controlled = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))

print(f"Training accuracy: {train_acc_controlled:.4f}")
print(f"Test accuracy:     {test_acc_controlled:.4f}")
print(f"Train-test gap:    {train_acc_controlled - test_acc_controlled:.4f}")


print("\n" + "=" * 70)
print("TASK 3: GINI VS ENTROPY")
print("=" * 70)

dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print(f"Gini test accuracy:    {test_acc_gini:.4f}")
print(f"Entropy test accuracy: {test_acc_entropy:.4f}")


print("\n" + "=" * 70)
print("TASK 4: RANDOM FOREST")
print("=" * 70)

random_forest = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
random_forest.fit(X_train_scaled, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, random_forest.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_clf_test, random_forest.predict(X_test_scaled))
rf_auc = roc_auc_score(y_clf_test, random_forest.predict_proba(X_test_scaled)[:, 1])

print(f"Training accuracy: {rf_train_acc:.4f}")
print(f"Test accuracy:     {rf_test_acc:.4f}")
print(f"Test ROC-AUC:      {rf_auc:.4f}")

feature_importances = pd.Series(
    random_forest.feature_importances_, index=X.columns
).sort_values(ascending=False)

print("\n--- Top 5 features by importance ---")
print(feature_importances.head(5))


print("\n" + "=" * 70)
print("TASK 4a: GRADIENT BOOSTING")
print("=" * 70)

gradient_boost = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
)
gradient_boost.fit(X_train_scaled, y_clf_train)

gb_train_acc = accuracy_score(y_clf_train, gradient_boost.predict(X_train_scaled))
gb_test_acc = accuracy_score(y_clf_test, gradient_boost.predict(X_test_scaled))
gb_auc = roc_auc_score(y_clf_test, gradient_boost.predict_proba(X_test_scaled)[:, 1])

print(f"Training accuracy: {gb_train_acc:.4f}")
print(f"Test accuracy:     {gb_test_acc:.4f}")
print(f"Test ROC-AUC:      {gb_auc:.4f}")


print("\n" + "=" * 70)
print("TASK 4b: FEATURE ABLATION STUDY")
print("=" * 70)

lowest_5_features = feature_importances.tail(5).index.tolist()
print("5 lowest-importance features (being removed):", lowest_5_features)

X_train_reduced = X_train_scaled.drop(columns=lowest_5_features)
X_test_reduced = X_test_scaled.drop(columns=lowest_5_features)

random_forest_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
random_forest_reduced.fit(X_train_reduced, y_clf_train)

rf_auc_full = rf_auc  # from Task 4, same model
rf_auc_reduced = roc_auc_score(y_clf_test, random_forest_reduced.predict_proba(X_test_reduced)[:, 1])

print(f"\nFull model AUC (all {X.shape[1]} features):        {rf_auc_full:.4f}")
print(f"Reduced model AUC ({X.shape[1]-5} features, 5 removed): {rf_auc_reduced:.4f}")


print("\n" + "=" * 70)
print("TASK 5: CROSS-VALIDATED COMPARISON")
print("=" * 70)

cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_to_compare = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Decision Tree (max_depth=5)': DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    'Random Forest': random_forest,
    'Gradient Boosting': gradient_boost,
}

cv_results = []
for model_name, model in models_to_compare.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv_splitter, scoring='roc_auc')
    cv_results.append({'Model': model_name, 'Mean CV AUC': scores.mean(), 'Std CV AUC': scores.std()})
    print(f"{model_name}: mean AUC = {scores.mean():.4f}, std = {scores.std():.4f}")

cv_results_table = pd.DataFrame(cv_results)


print("\n" + "=" * 70)
print("TASK 6: HYPERPARAMETER TUNING WITH GRIDSEARCHCV")
print("=" * 70)

tuning_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

grid_search = GridSearchCV(
    tuning_pipeline, param_grid, cv=cv_splitter, scoring='roc_auc', n_jobs=-1
)

start_time = time.time()
grid_search.fit(X_train, y_clf_train)
elapsed = time.time() - start_time

print(f"Grid search finished in {elapsed:.1f} seconds")
print("Best parameters found:", grid_search.best_params_)
print(f"Best cross-validated AUC: {grid_search.best_score_:.4f}")

best_pipeline = grid_search.best_estimator_
test_auc_best_pipeline = roc_auc_score(y_clf_test, best_pipeline.predict_proba(X_test)[:, 1])
print(f"Test-set AUC of the best pipeline: {test_auc_best_pipeline:.4f}")

n_grid_points = 3 * 3 * 2  # 3 n_estimators x 3 max_depth x 2 min_samples_leaf choices
print(f"\nTotal grid points: {n_grid_points}. With 5-fold CV, that's "
      f"{n_grid_points} x 5 = {n_grid_points * 5} model fits in total.")


print("\n" + "=" * 70)
print("TASK 7: MANUAL LEARNING CURVE")
print("=" * 70)


learning_curve_rows = []
for fraction in [0.2, 0.4, 0.6, 0.8, 1.0]:
    n_rows = int(fraction * len(X_train))
    X_subset = X_train.iloc[:n_rows]
    y_subset = y_clf_train.iloc[:n_rows]

    # clone() makes a fresh, untrained copy with the same settings,
    # so each fraction starts training from scratch
    pipeline_copy = clone(best_pipeline)
    pipeline_copy.fit(X_subset, y_subset)

    train_auc = roc_auc_score(y_subset, pipeline_copy.predict_proba(X_subset)[:, 1])
    test_auc = roc_auc_score(y_clf_test, pipeline_copy.predict_proba(X_test)[:, 1])

    learning_curve_rows.append({
        'Training fraction': fraction,
        'Rows used': n_rows,
        'Training AUC': train_auc,
        'Test AUC': test_auc
    })

learning_curve_table = pd.DataFrame(learning_curve_rows)
print(learning_curve_table.round(4).to_string(index=False))


print("\n" + "=" * 70)
print("TASK 8: SERIALIZE THE BEST MODEL")
print("=" * 70)

joblib.dump(best_pipeline, 'best_model.pkl')
print("Saved best pipeline to 'best_model.pkl'")

loaded_model = joblib.load('best_model.pkl')

new_patients = pd.DataFrame([
    {  # Patient A: older, higher blood pressure, smoker, family history
        'Age': 68, 'BMI': 31.5, 'SystolicBP': 155, 'DiastolicBP': 95,
        'GlucoseMgDl': 140, 'RestingHR': 82, 'ExerciseHrsPerWeek': 0.5,
        'Sex_Male': True, 'Smoker_Yes': True,
        'BloodType_A-': False, 'BloodType_AB+': False, 'BloodType_AB-': False,
        'BloodType_B+': False, 'BloodType_B-': False, 'BloodType_O+': True,
        'BloodType_O-': False, 'FamilyHistory_Yes': True, 'Diabetes_Yes': True
    },
    {  # Patient B: younger, healthier numbers, non-smoker, no family history
        'Age': 29, 'BMI': 22.0, 'SystolicBP': 112, 'DiastolicBP': 72,
        'GlucoseMgDl': 88, 'RestingHR': 64, 'ExerciseHrsPerWeek': 5.0,
        'Sex_Male': False, 'Smoker_Yes': False,
        'BloodType_A-': False, 'BloodType_AB+': False, 'BloodType_AB-': False,
        'BloodType_B+': True, 'BloodType_B-': False, 'BloodType_O+': False,
        'BloodType_O-': False, 'FamilyHistory_Yes': False, 'Diabetes_Yes': False
    }
])[X.columns]  # reorder columns to exactly match training data

predictions = loaded_model.predict(new_patients)
probabilities = loaded_model.predict_proba(new_patients)[:, 1]

print("\n--- Reload-and-predict test ---")
for i, (pred, proba) in enumerate(zip(predictions, probabilities)):
    label = 'Yes' if pred == 1 else 'No'
    print(f"Patient {chr(65+i)}: predicted HeartDisease = {label} "
          f"(probability = {proba:.3f})")


print("\n" + "=" * 70)
print("SUMMARY TABLE: ALL MODELS")
print("=" * 70)

summary_rows = [
    {'Model': 'Logistic Regression', 'Mean CV AUC': cv_results_table.iloc[0]['Mean CV AUC'],
     'Std CV AUC': cv_results_table.iloc[0]['Std CV AUC'], 'Test AUC': 0.7785},  # from Part 3 basic
    {'Model': 'Decision Tree (max_depth=5)', 'Mean CV AUC': cv_results_table.iloc[1]['Mean CV AUC'],
     'Std CV AUC': cv_results_table.iloc[1]['Std CV AUC'], 'Test AUC': test_acc_controlled},
    {'Model': 'Random Forest', 'Mean CV AUC': cv_results_table.iloc[2]['Mean CV AUC'],
     'Std CV AUC': cv_results_table.iloc[2]['Std CV AUC'], 'Test AUC': rf_auc},
    {'Model': 'Gradient Boosting', 'Mean CV AUC': cv_results_table.iloc[3]['Mean CV AUC'],
     'Std CV AUC': cv_results_table.iloc[3]['Std CV AUC'], 'Test AUC': gb_auc},
    {'Model': 'Tuned Random Forest (GridSearchCV)', 'Mean CV AUC': grid_search.best_score_,
     'Std CV AUC': np.nan, 'Test AUC': test_auc_best_pipeline},
]
summary_table = pd.DataFrame(summary_rows)
print(summary_table.round(4).to_string(index=False))



ModuleNotFoundError: No module named 'numpy'